In [1]:
# CELL 1 - Download cloudflared
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

In [2]:
import torch
import gc
import gradio as gr
from diffusers import PixArtAlphaPipeline
from peft import PeftModel
from transformers import T5Tokenizer

# CONFIG
BASE_MODEL  = "PixArt-alpha/PixArt-XL-2-1024-MS"
HF_REPO     = "arpita2desh/pixart-hands-lora"
MERGED_REPO = "arpita2desh/pixart-hands-lora-merged"
DEVICE      = "cuda"

pipe = None
current_model = None


def clear():
    gc.collect()
    torch.cuda.empty_cache()


def load_model(name):
    global pipe, current_model
    if current_model == name:
        return pipe

    # Unload previous
    if pipe is not None:
        del pipe
        pipe = None
        clear()

    print(f"Loading {name}...")

    if name == "Base":
        pipe = PixArtAlphaPipeline.from_pretrained(
            BASE_MODEL, torch_dtype=torch.float16
        ).to(DEVICE)

    elif name == "Checkpoint-500":
        pipe = PixArtAlphaPipeline.from_pretrained(
            BASE_MODEL, torch_dtype=torch.float16
        ).to(DEVICE)
        pipe.transformer = PeftModel.from_pretrained(
            pipe.transformer, HF_REPO, subfolder="checkpoint-500"
        )

    elif name == "Checkpoint-Final":
        pipe = PixArtAlphaPipeline.from_pretrained(
            BASE_MODEL, torch_dtype=torch.float16
        ).to(DEVICE)
        pipe.transformer = PeftModel.from_pretrained(
            pipe.transformer, HF_REPO, subfolder="checkpoint-final"
        )

    elif name == "Merged":
        pipe = PixArtAlphaPipeline.from_pretrained(
            MERGED_REPO, torch_dtype=torch.float16, tokenizer=None
        ).to(DEVICE)
        pipe.tokenizer = T5Tokenizer.from_pretrained("t5-base")

    current_model = name
    return pipe


def generate(model_name, prompt):
    pipe = load_model(model_name)
    generator = torch.Generator(device=DEVICE).manual_seed(42)
    with torch.no_grad():
        image = pipe(
            prompt=prompt,
            negative_prompt="extra fingers, missing fingers, bad anatomy",
            num_inference_steps=30,
            guidance_scale=7.5,
            height=512,
            width=512,
            generator=generator,
        ).images[0]
    return image


demo = gr.Interface(
    fn=generate,
    inputs=[
        gr.Dropdown(
            choices=["Base", "Checkpoint-500", "Checkpoint-Final", "Merged"],
            value="Checkpoint-Final",
            label="Select Model"
        ),
        gr.Textbox(label="Prompt"),
    ],
    outputs=gr.Image(label="Generated Image"),
    title="PixArt Hands"
)

demo.launch(share=True)

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://11036ec8a76513987a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [3]:
import requests, threading

RENDER_URL = "https://kaggle-proxy-561j.onrender.com"
SECRET = ""

# Launch with share=True to get a public gradio.live URL
result = demo.launch(
    server_name="0.0.0.0",
    server_port=7860,
    share=True,
    prevent_thread_lock=True
)

# Register the share URL with Render
share_url = result[2]  # gradio returns (local_url, local_url, share_url)
print(f"Share URL: {share_url}")

r = requests.post(f"{RENDER_URL}/update", json={
    "url": share_url,
    "secret": SECRET
})
print(f"Registered: {r.json()}")
print(f"\n🎉 PERMANENT URL: {RENDER_URL}")

Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
* Running on public URL: https://11036ec8a76513987a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Share URL: https://11036ec8a76513987a.gradio.live
Registered: {'ok': True, 'url': 'https://11036ec8a76513987a.gradio.live'}

🎉 PERMANENT URL: https://kaggle-proxy-561j.onrender.com
